# Jute-App — Notebook as Application Container

**Status:** draft · **Date:** 2026-06-01 · **Owner:** notebook platform

## Vision

A *Jute-App* is a notebook that **runs like an application**. Opening it boots an app shell: **frontend cells** render the UI, the **reactive DAG engine** drives recomputation underneath, and cells communicate over an **internal `ipc://` ZeroMQ control bus**. The `.ipynb` file is the app manifest — its cells, port wiring, and which cells are `frontend` define the running application.

This spec covers the architecture, the reactive loop, the UI/UX in both *Edit* and *App* modes, the container lifecycle, and an incremental migration path. It is grounded in the code as it exists today (`crates/spur-notebook`).

> **Non-goal:** this is *not* a headless server runtime. The frontend is first-class — the notebook container always has a UI surface (the Jute Tauri app), even if some deployments later hide it.

## 1. Concept: three layers, one container

| Layer | Role | Substrate (today → target) |
|---|---|---|
| **Frontend cells** | Presentation — live views & inputs bound to DAG state | Jupyter **comm** protocol (`CommOpen`/`CommMessage`) + port MIME preview → two-way bound widgets |
| **Reactive DAG engine** | The brain — dependency tracking, topological cascade, failure propagation | In-process `ReactiveEngine` (Tokio task) — unchanged role, gains a bus broker |
| **Internal cell bus** | Control plane — cell↔cell↔engine signalling | *new* `ipc://` ZeroMQ pub/sub, scoped per notebook id |

The **data plane stays on disk**: cells exchange bulk data as Arrow IPC files in `~/.spur/notebooks/<id>/ports/`, with `manifest.json` as the single source of truth. The bus carries only lightweight control events (pointers + signals), never payloads.

> **One identity to rule them all.** Container, port store, and bus socket are all keyed by `notebook_id_for_path` (`ports.rs:11`). The notebook id *is* the app instance.

## 2. Where we are today (grounded)

The DAG backbone is **in-process Rust**, not a bus. ZeroMQ appears only as the per-kernel Jupyter execution transport. The engine learns about port writes by **reading `manifest.json` after the runner reports completion** — `iopub` display is for the GUI only.

```mermaid
flowchart TB
  subgraph FE["Jute Frontend (Tauri)"]
    UI["Cell views render MIME / display_data"]
  end
  subgraph HOST["Host process — in-process orchestration"]
    STORE["NotebookStore<br/>broadcast deltas"]
    ENG["ReactiveEngine (Tokio task)<br/>DAG + topological cascade"]
    RUN["CellRunner → run_cell MCP tool"]
  end
  PY["python3 kernel"]
  DISK[("ports/ — Arrow IPC + manifest.json")]

  UI -->|run cell / push source| STORE
  STORE -->|mpsc SourcePush + deltas| ENG
  ENG --> RUN
  RUN -->|Jupyter wire ZMQ tcp| PY
  PY -->|spur.put writes payload| DISK
  ENG -->|reads manifest after run| DISK
  PY -->|display MIME via iopub| UI
```

**Key code anchors**
- `ReactiveEngine` — `crates/spur-notebook/src/dag/engine.rs:201`
- Cascade (topological) — `engine.rs:305-347`
- Run loop (mpsc + broadcast) — `spawn_reactive_engine`, `engine.rs:570-631`
- `CellRunner` execution seam — `engine.rs:106`
- Port store / bootstraps — `jute-notebook/src-tauri/src/ports.rs`
- Comm types (frontend substrate) — `wire_protocol.rs:513-536`
- Jupyter ZMQ driver (kept as-is) — `backend/wire_protocol/driver_zeromq.rs`

## 3. Target architecture

Promote the engine's **already-existing event taxonomy** (`emit_dag_status_changed`, `emit_run_report`, `SourcePush`) onto a dedicated `ipc://` control bus, with the **engine as central broker**. Frontend cells bind to ports via comm. The Jupyter wire and the on-disk data plane are unchanged beneath.

```mermaid
flowchart TB
  subgraph APP["Jute-App Container — one notebook id"]
    subgraph FEL["Frontend cells (UI layer)"]
      FCC["chart — bound to port 'forecast'"]
      FCS["slider — writes port 'horizon'"]
    end
    subgraph ENGINE["Reactive Engine"]
      DAG["DAG authority<br/>topological cascade"]
      BROKER["ipc:// bus broker<br/>XPUB/XSUB"]
    end
    PY["python kernel<br/>+ spur bus client"]
    DENO["deno kernel<br/>+ spur bus client"]
    DISK[("ports/ — Arrow IPC + manifest.json<br/>data plane · source of truth")]
  end

  FCS -->|source.push via comm| DAG
  DAG --> BROKER
  BROKER -->|cell.run.request| PY
  BROKER -->|cell.run.request| DENO
  PY -->|writes payload| DISK
  DENO -->|writes payload| DISK
  PY -->|port.updated control| BROKER
  DENO -->|port.updated control| BROKER
  BROKER -->|port.updated| DAG
  DAG -->|render update via comm| FCC
```

> Beneath this, each kernel still talks the **Jupyter wire ZMQ** for code execution (`driver_zeromq.rs`). That is a *separate* socket set from the cell bus — see §4.

## 4. Three buses — do not fuse them

The most important design rule. "Internal IPC ZeroMQ bus" is a **third, new role** — not the Jupyter driver, not the comm channel.

| Bus | Role | Transport | Auth | Status |
|---|---|---|---|---|
| **Jupyter wire** | host → one kernel: execute code | `tcp://127.0.0.1:*`, 5 sockets | HMAC per-kernel key | exists (`driver_zeromq.rs`) |
| **Comm / Tauri events** | frontend cell ↔ engine: render + user input | Tauri IPC / comm msgs | in-process trust | partial (`CommOpen`/`CommMessage`) |
| **Internal cell bus** | cell ↔ cell ↔ engine: control events | **`ipc://` unix socket** per notebook | filesystem perms (+ optional token) | **new** |

**Why a separate bus, not the Jupyter driver?** The Jupyter driver is a *per-kernel, HMAC-signed, 5-socket RPC client*. Forcing external participants to speak that envelope just to say "port X bumped" is the wrong weight class. The cell bus is a minimal pub/sub.

**Why `ipc://` not `tcp://`?** It is local ("in the notebook container"), lower overhead, and filesystem-scoped — the socket lives at `~/.spur/notebooks/<id>/bus.sock`, inheriting the same per-notebook isolation as the port store.

### Bus event taxonomy (control plane only)

```
port.updated      { port, version, schema, writer }   # emitted on every spur.put
cell.run.request  { cell_id, expected_version }
cell.status       { cell_id, running | ok | failed | upstream_failed }
source.push       { port, ipc_ref }                    # external / frontend input
dag.status.changed{ states, port_manifest }            # mirrors publish_dag_status_changed
```

Payloads never ride the bus — `port.updated` carries a *reference*; consumers read the Arrow file from disk.

## 5. The reactive loop (what makes it an *app*)

User interaction in a frontend cell flows through the DAG and back to the UI — a closed loop. This is the difference between a notebook and an application.

```mermaid
sequenceDiagram
  actor User
  participant FCS as Frontend cell · slider
  participant ENG as Reactive Engine (broker)
  participant BUS as ipc:// bus
  participant K as Cell worker (kernel)
  participant DISK as Port store (Arrow disk)
  participant FCC as Frontend cell · chart

  User->>FCS: drag slider (horizon = 12)
  FCS->>ENG: source.push { port: horizon }
  ENG->>ENG: rebuild graph · topo-sort downstream
  ENG->>BUS: cell.run.request { forecast_cell }
  BUS->>K: cell.run.request
  K->>DISK: spur.put("forecast", arrow)
  K->>BUS: port.updated { forecast, v=n }
  BUS->>ENG: port.updated
  ENG->>DISK: read manifest (authoritative version)
  ENG->>FCC: render update { forecast }
  Note over FCC: chart re-renders live
```

The cascade ordering, the `upstream_failed` propagation, and staleness coalescing all stay inside the engine (`engine.rs:305-347`) — the bus only transports the requests out and the signals back.

## 6. Frontend cells — comm-bound live views over ports

A **frontend cell** is not a new execution model; it is a UI component **bound to one or more ports** through the comm channel. It generalizes today's static port MIME preview (`ports.rs:74` / `:406`) into a live, two-way binding.

```mermaid
flowchart LR
  PIN[("port: forecast<br/>Arrow + schema")] -->|on bump → render(value)| W["Frontend cell widget"]
  W -->|on user input → source.push(value)| POUT[("port: horizon")]
```

### Declaration

A cell becomes a frontend cell via metadata (reusing the `spur` cell-metadata facet pattern, like `dag` and `code_type`):

```jsonc
{
  "spur": {
    "frontend": {
      "kind": "chart",            // chart | table | slider | form | html | custom
      "binds": ["forecast"],      // input ports → render
      "emits": ["horizon"],       // user actions → source.push to these ports
      "props": { "x": "month", "y": "revenue" }
    }
  }
}
```

### Two flavors
- **View cells** (`binds`, no `emits`): pure outputs — charts, tables, KPIs. Re-render on `port.updated`.
- **Control cells** (`emits`): inputs — sliders, dropdowns, buttons, forms. User action → `source.push` → cascade.

### Rendering substrate
- A control cell's code (Deno/Python) registers a comm and a render target; the **frontend** owns the actual widget. The kernel side just declares the binding and pushes initial state.
- View cells need **no kernel at all** for re-render — the engine pushes the port value to the frontend over comm when the manifest version changes. Cheap, and survives kernel restarts.

## 6a. Live-render transport — Option A + Option B (the hybrid)

§6 says a frontend view cell "re-renders on `port.updated`." This section pins down **how the new bytes actually reach a mounted widget without re-running the cell or reloading its output** — and why the two mechanisms (A and B) are *layers of one pipe*, not alternatives.

### The axis that matters: HTML-snapshot vs. persistent-widget

There are two ways a frontend shows output, and they behave differently on update:

| Model | Update mechanism | On a port bump | Widget state (Perspective WASM, pivots) |
|---|---|---|---|
| **HTML snapshot** | `display_data` / `update_display_data` carry an HTML blob | the whole output HTML is **replaced** → iframe `srcDoc` swap → full reload | **lost** (re-bootstrap every tick) |
| **Persistent widget** | a long-lived JS object receives a *message* and mutates itself (`table.update()`) | DOM patched in place, **no HTML swap** | **kept** |

> **`iframe`-vs-`inline-DOM` is the wrong lever.** Following JupyterLab's inline rendering does **not** give live stateful updates: `update_display_data` still replaces the output HTML, so a `<perspective-viewer>` is destroyed and re-created either way. Inline DOM also (a) **breaks script-driven artifacts** — JupyterLab strips `<script>` from untrusted HTML (DOMPurify) and `innerHTML`-injected scripts don't execute even when trusted; our dashboard only runs because the sandboxed iframe has `allow-scripts` — and (b) is a **security regression**: inline = same-origin as the Tauri app, so kernel-authored HTML could reach Tauri IPC / other notebooks. The sandbox at `OutputView.tsx:94` (`allow-scripts` **without** `allow-same-origin`) is a deliberate isolation boundary, and Jute has not built the Jupyter trust/signature model that would make same-origin output safe (`OutputView.tsx:132` TODO). JupyterLab's *live widgets* come from ipywidgets/**comm** (persistent widget + messages), not from its inline HTML rendering. So the right axis is **snapshot → persistent-widget**, keeping the sandbox.

### Option A — persistent sandboxed widget + `postMessage` (the transport leaf)

Keep the sandboxed output iframe; stop treating it as a disposable snapshot. The widget boots Perspective **once** (indexed table), then receives Arrow deltas over a narrow `postMessage` channel and calls `table.update()` in place. Jute already runs a `postMessage` bridge for the height reporter (`OutputView.tsx:105-124` listener, `:212-228` injected reporter) — A just makes it **bidirectional**.

**Three `OutputView.tsx` changes turn snapshot output into a live widget (untagged output is unchanged — fully backward compatible):**
1. **Tag live outputs** — producer sets `metadata: { "application/vnd.spur.live+json": { port } }`; `OutputView` reads it to choose the live path.
2. **Mount once** — for a live display, build `srcDoc` from the **first** payload only and key the iframe by `display_id`, not by `html` (today `srcDoc = useMemo(…, [html])` at `:99` + `srcDoc={srcDoc}` at `:131` remount on every change).
3. **Relay, don't re-render** — when the store would rewrite that display's data (the `update_display_data` path, `:789-806`/`:801`), instead `iframeRef.current.contentWindow.postMessage({ source:"spur-port", kind:"update", arrowB64 }, "*")`. The widget's listener does `table.update()`.

**Widget message contract (the stable A/B seam):**
```
host → iframe:  { source:"spur-port", kind:"load"|"update", port, version, arrowB64 }
iframe → host:  { source:"spur-port", kind:"ready", port }   // mounted; request initial/live feed
```

### Option B — the comm bus chooses the *trigger*

B (from §4) is the control plane: the engine detects a manifest bump and emits `port.updated {port, version, ref}` — a **notification with a reference, never the payload**. B answers *"what changed and who needs to know."* A answers *"how the bytes land in the widget."*

### The hybrid: B drives A

They compose — B is the spine, A is the last mile. The widget's `postMessage` contract is the boundary, so the **driver behind step 3 is swappable** without touching the widget or the relay:

```mermaid
flowchart TB
  subgraph ENG["Reactive Engine — control plane (B)"]
    DET["detect manifest bump"]
    PU["emit port.updated { port, version, ref }"]
  end
  subgraph HOST["Jute host — the A/B seam (relay)"]
    SUB["ipc:// bus subscriber"]
    READ["read Arrow file by ref"]
    POST["iframe.postMessage({ kind:'update', arrowB64 })"]
  end
  subgraph IF["Sandboxed output iframe — transport leaf (A)"]
    LIS["message listener"]
    UPD["table.update(arrow) — upsert by index"]
    VIEW["perspective-viewer redraws · WASM + pivots kept"]
  end
  DISK[("ports/ — Arrow IPC + manifest.json · source of truth")]
  DET --> PU --> SUB --> READ --> POST --> LIS --> UPD --> VIEW
  READ -. reads payload .-> DISK
  IF -. 'ready' handshake .-> HOST
```

```mermaid
sequenceDiagram
  participant K as Producer cell / external writer
  participant DISK as Port store (Arrow disk)
  participant ENG as Engine (bus broker · B)
  participant HOST as Jute host relay
  participant IF as Sandboxed widget (A)
  Note over IF: booted ONCE — Perspective table loaded + indexed
  K->>DISK: spur.put("weather", arrow)  → version n+1
  DISK-->>ENG: manifest bump
  ENG->>HOST: port.updated { weather, n+1, ref }   (control only, no bytes)
  HOST->>DISK: read weather@v(n+1).arrow by ref
  HOST->>IF: postMessage { kind:"update", arrowB64 }
  IF->>IF: table.update(arrow)  (upsert by index)
  Note over IF: redraw in place · WASM kept · pivot/sort kept · cell NOT re-run
```

**What differs between A-only and A+B is only what fires the relay:**

| | Trigger of the relay (step 3) | Cost |
|---|---|---|
| **A-only** (ship first) | local — the `update_display_data` store path, or a kernel poll | de-risks the channel; crude trigger |
| **A + B** (end state) | the **comm bus** `port.updated` subscriber reads the ref off disk | no kernel-blocking loop, no poll, scales to any process |

The iframe-facing contract is **identical** in both, so the host relay is written once (A) and B later swaps the trigger underneath it.

### Grounding & prototype findings

- **The updatable-display mechanism is already wired:** `display_data` + `transient.display_id` registers a slot (`stores/notebook.ts:751`); `update_display_data` rewrites it in place (`:789-806`). Verified: a single execution re-rendered the same output as an **external** writer bumped `weather` v5→v6→v7 (1461→1000→500 rows) — no cell re-run.
- **A background watcher that outlives the cell run does NOT surface updates today** — post-run iopub is not routed back to the cell output. This is exactly why the live path must be **B's comm bus**, not a reliance on post-run display messages. (Empirically confirmed: the port hit v5 on disk while the watcher's output stayed at v4.)
- **`table.update()` streaming is verified inside the Jute sandbox** (500k rows, d3fc) — so A's in-place path is viable; only the host→iframe relay (changes 1–3 above) is missing.

### Migration fit (slots into §10)

- **Step 1 (Frontend-cell MVP):** add changes 1–3 to `OutputView.tsx` + the `application/vnd.spur.live+json` tag → a persistent widget driven by the local trigger. **This is Option A and is independently shippable.**
- **Step 4 (closed loop):** point the relay's trigger at B's `port.updated` subscriber → **the hybrid**. No change to the widget or the relay contract.


## 6b. Relay variants — Design 1 vs Design 2 (ADR)

§6a's relay has two implementations. Both honor invariant #2 (the **bus** carries references only); they differ in **who turns the ref into bytes** and **how the bytes cross the sandbox wall** — forced by the fact that the widget's iframe has no filesystem, so it cannot dereference a disk path itself.

### Design 1 — host dereferences, pushes transferable bytes

```mermaid
flowchart LR
  ENGINE["Engine (B): emits port.updated + ref"]
  subgraph HOST["Host — fs-capable (Rust + main webview JS)"]
    RUST["Rust: mmap read · zero-copy"]
    MJS["main-window JS heap"]
  end
  subgraph IFRAME["Sandboxed iframe — widget (A)"]
    WK["Perspective worker"]
    ENG["WASM engine columns"]
  end
  DISK[("ports dir · weather@vN.arrow")]
  ENGINE -->|ref| RUST
  RUST -->|"Tauri IPC · COPY 1"| MJS
  MJS -->|"postMessage transfer · zero-copy"| WK
  WK -->|"ingest · COPY 2"| ENG
  RUST -.mmap.-> DISK
```

### Design 2 — ref forwarded, widget fetches

```mermaid
flowchart LR
  ENGINE["Engine (B): emits port.updated + ref"]
  subgraph HOST["Host"]
    RELAY["Relay: forward ref only · no payload"]
    SRV["Local endpoint · serves port files"]
  end
  subgraph IFRAME["Sandboxed iframe — widget (A)"]
    FET["fetch(ref) → iframe heap · COPY 1"]
    WK["Perspective worker"]
    ENG["WASM engine columns"]
  end
  DISK[("ports dir · weather@vN.arrow")]
  ENGINE -->|ref| RELAY
  RELAY -->|"postMessage ref · no payload"| FET
  FET -->|"transfer · zero-copy"| WK
  WK -->|"ingest · COPY 2"| ENG
  SRV -.serves.-> DISK
  FET -.HTTP.-> SRV
```

### They cost the same memory — the copy floor is identical

| Hop | Design 1 | Design 2 |
|---|---|---|
| disk → host | `mmap` · zero-copy | `mmap` · zero-copy |
| land bytes in browser heap | Tauri IPC into main JS · **COPY 1** | `fetch()` into iframe heap · **COPY 1** |
| browser → Perspective worker | transferable · zero-copy | transferable · zero-copy |
| Perspective ingest (WASM columns) | bulk columnar copy · **COPY 2** | bulk columnar copy · **COPY 2** |
| **total copies** | **2** | **2** |

> **Perspective's Arrow ingest is identical in both.** It cannot tell which path delivered the buffer; both pay the same single WASM-heap ingest (the copy floor for any WASM engine — it owns mutable, indexed columns and cannot alias immutable Arrow buffers). Arrow IPC is still the right choice in both: no re-encode, types preserved, `memcpy`-class ingest instead of a JSON/records parse. **So the choice is not about memory or Arrow efficiency — it is about transport topology.**

### Pros / cons

**Design 1 — host-deref + transferable `postMessage`**
- ✅ **Self-contained** — no server, no extra origin, no CORS.
- ✅ Smallest change — extends the existing `OutputView.tsx` `postMessage` bridge; ships as migration step 1.
- ✅ Same copy count as Design 2; transferable hop is zero-copy.
- ⚠️ Payload **transits the host main-window JS heap** → transient residency + GC pressure for large buffers.
- ⚠️ Tauri IPC marshals a **complete** buffer (no streaming).

**Design 2 — ref-forward + widget `fetch()`**
- ✅ **Forward-only relay** — payload never touches host JS; reference discipline reaches the widget.
- ✅ **Streams** large/continuous feeds (chunked fetch); confines data to iframe + worker.
- ✅ Scales to big ports without pressuring the host process.
- ❌ **Not self-contained** — needs a local serving endpoint (`spur-port` asset protocol or loopback) + opaque-origin CORS handling.
- ❌ More moving parts; a sandbox without `allow-same-origin` fetching a custom scheme is fiddly.

### ADR-006 — Live-view delta transport into the sandboxed widget

- **Status:** Accepted (Design 1 as default; Design 2 deferred, opt-in).
- **Context:** A frontend view cell must receive Arrow deltas live (no cell re-run, no iframe reload). The widget runs in a sandboxed, opaque-origin iframe with **no filesystem**, so it cannot dereference a port path itself. The control bus (B) is reference-only. Perspective's WASM ingest copy is unavoidable and identical regardless of delivery path, so memory/Arrow-efficiency does **not** discriminate between the options — only transport topology, self-containment, and payload scale do.
- **Decision:** Adopt **Design 1** (host dereferences via `mmap`, pushes the Arrow buffer to the iframe as a **Transferable `ArrayBuffer`** over `postMessage`) as the **default and first to ship**. Keep **Design 2** (relay forwards the ref unchanged; widget `fetch()`es from a local port-serving endpoint) as an **opt-in** for large or continuously-streaming ports. Both share one envelope — `{ source:"spur-port", kind:"load"|"update", port, version }` — differing only in whether the message carries `arrowB64`/transferable bytes (D1) or a `ref` URL (D2), so the widget and the bus are unchanged when switching.
- **Consequences:**
  - ➕ Ships self-contained with zero new infrastructure; equal on the copy floor; de-risks the channel before B's bus exists.
  - ➕ The shared envelope means D1→D2 is a transport swap, not a redesign — chosen per output (or per payload-size threshold), not globally.
  - ➖ D1's host-heap transit caps comfortable payload size; crossing that threshold requires standing up D2's endpoint (server + CORS).
  - ➖ Neither reaches literal zero-copy into the grid — the Perspective ingest copy is the accepted floor; eliminating it would mean abandoning Perspective's pivot/update engine.
- **Revisit when:** typical delta size routinely stresses the host main-window heap, or a local port-serving protocol lands for other reasons (then promote Design 2 to default for large ports).


## 6c. Why the copy floor exists — first principles (and why fs/mmap access does NOT remove it)

§6a/§6b assume a 2-copy floor. This pins down *why*, because the obvious "fix" — give the widget filesystem/mmap access so it reads the Arrow file directly — **does not work**, and understanding why prevents a costly dead end.

### Two distinct things Arrow's "zero-copy" means
1. **No parse / no re-encode** — the on-disk bytes *are* the columnar layout; any language interprets them with no deserialization. **Process- and language-agnostic. Preserved everywhere — including into the browser and into Perspective.** This is most of Arrow's value, and we keep all of it.
2. **Zero-copy *aliasing*** — a consumer builds array **views that point into existing memory** instead of copying. Across processes this is free **only via shared memory** (`mmap` of the same file, or `shm`), and **only for runtimes that can construct array views over a foreign pointer** (Rust / C++ / pyarrow). It is *not* "bytes over a socket without a copy."

> **Correction to an earlier framing:** a *process boundary* is not inherently a copy — Arrow crosses process boundaries zero-copy all the time via shared memory. The copy in our path is forced at one specific place: **entry into the JS/WASM runtime.**

### The real blocker: the JS/WASM memory model — not the sandbox, not the process boundary
- **JS `ArrayBuffer` cannot be backed by foreign memory.** No API mmap-aliases a file region (or an external `shm` pointer) into an `ArrayBuffer`; the engine owns its heap, so a read *fills a JS-owned buffer by copy*. `SharedArrayBuffer` is also engine-allocated — you cannot adopt an external mapping into it.
- **WASM linear memory is a closed, engine-owned region.** Data is visible to Perspective's WASM only once copied *into* that linear memory; you cannot mount an mmap'd file as part of it (true even for WASI file access — a WASM module reads *into* its linear memory).
- **Perspective then copies again** into its mutable, indexed column store (it cannot alias immutable Arrow buffers — it supports `update`/pivot/filter).

**Proof it is the runtime, not fs access:** the **Deno kernel has full filesystem access and no sandbox**, yet `Deno.readFileSync` still **copies** disk → a JS `Uint8Array` (apache-arrow then views *that copy* zero-copy). Adding fs access changed nothing about the copy.

### Two independent problems — fs access solves only the first
| Problem | What it is | Does fs/mmap access solve it? |
|---|---|---|
| **1 · Access (transport channel)** | who reads the file + how bytes reach the widget (relay vs fetch — see §6b) | **Yes** — but at the cost of a sandbox escape (kernel-authored HTML with raw disk access) |
| **2 · Copy (zero-copy)** | the JS/WASM runtime can't alias foreign memory | **No** — the copy is the runtime, not the permission |

So "solve mmap for the widget" removes the relay/server **and tears a hole in the isolation boundary** — while delivering **zero** copy reduction. Net negative.

### The zero-copy frontier

```mermaid
flowchart LR
  DISK[("Arrow IPC file")]
  subgraph NATIVE["Producer address space (Rust / Python / C++) — zero-copy zone"]
    HOST["mmap → Arrow Buffer · views, no copy (ports.rs)"]
  end
  subgraph BROWSER["Browser JS + WASM — cannot alias foreign memory"]
    JS["JS ArrayBuffer"]
    WK["Perspective worker"]
    ENG["WASM engine columns"]
  end
  DISK -. "mmap · zero-copy" .-> HOST
  HOST ==>|"COPY 1 · JS/WASM entry — no foreign alias"| JS
  JS -->|"transfer · zero-copy"| WK
  WK ==>|"COPY 2 · engine ingest — mutable columns"| ENG
```

Zero-copy holds from disk through the **producer's address space** (this is exactly why Python `spur.put` ↔ Deno `spur.get` interoperate with no re-encode). It ends at the **JS/WASM entry** — the thick edges are the two unavoidable copies.

### What would actually eliminate a copy (and why we don't)
- **Drop COPY 1:** host the engine **natively** in the producer's address space (e.g. native Perspective over an mmap'd pyarrow table). But Perspective-in-Jute is **WASM-in-a-browser-iframe** by design — that's what we're choosing it for.
- **Drop COPY 2:** render straight from apache-arrow views (canvas / Plot) with **no mutable engine** — losing pivot / filter / `update`, i.e. the reason to use Perspective.

Either path means *not Perspective, not in a browser* — a different product. The copies are the price of the boundary between the producer's memory and an **interactive WASM analytics engine**; you can relocate that boundary or accept it, not both.

### Reframe — the floor is cheap; stop chasing it
Arrow already eliminated the **expensive** cost (serialization, parsing, per-cell allocation, type re-inference), and that win is preserved end-to-end. What remains are two **O(bytes) `memcpy`** of small streaming **delta** RecordBatches (KB per `update`) — constant-factor and not a bottleneck. It is not worth trading the sandbox or rearchitecting around a native engine to remove them.

### Decision impact
- **Confirms the ADR (§6b):** Design 1 default, **sandbox kept**, Arrow IPC end-to-end.
- **Rejected alternative — "give the widget filesystem/mmap access":** solves access (Problem 1) at a security cost and yields **no** zero-copy (Problem 2 is the JS/WASM runtime). Explicitly out.
- **Mental model:** *fs access is about **reaching** the bytes; zero-copy is about **aliasing** them. The browser may be allowed to reach, but never to alias — so the copy stays.*


## 7. UI/UX — two modes of one document

A Jute-App notebook is viewed in one of two modes, toggled in the title bar. Same file; different surface.

- **Edit (Studio) mode** — the full notebook: code, DAG wiring badges, per-cell run controls, *and* inline frontend-cell previews. For authors.
- **App (Run) mode** — only frontend cells, laid out as an application. Code is hidden; the document behaves like a deployed app. For users.

```mermaid
stateDiagram-v2
  [*] --> Edit
  Edit --> App: ▶ Run App
  App --> Edit: ✎ Edit
  state Edit { [*] --> code_and_widgets }
  state App  { [*] --> widgets_only }
```

### 7a. Edit (Studio) mode — wireframe

Code cells show DAG badges (`produces`/`consumes`), per-cell run, and an inline preview of any frontend binding.

```text
┌────────────────────────────────────────────────────────────────────────┐
│  forecast.ipynb           [ Edit ▾ ] [ ▶ Run App ]      ● python ● deno  │
├────────────────────────────────────────────────────────────────────────┤
│ [py]   ⬡ produces: sales                                      v3   ▶ run │
│  import pandas as pd;  spur.put("sales", load())                         │
│  └─ ✓ port sales · v3 · 1,240 rows × 4 cols                             │
├────────────────────────────────────────────────────────────────────────┤
│ [py]   ⬡ consumes: sales, horizon   produces: forecast        v7   ▶ run │
│  out = fit(spur.get("sales"), spur.get("horizon"))                       │
│  spur.put("forecast", out)                                              │
│  └─ ◫ FRONTEND VIEW  (chart · binds: forecast)                          │
│     ┌──────────────────────────────────────────────┐                    │
│     │  ▁▂▃▅▆▇█▇▆▅   revenue projection              │                    │
│     └──────────────────────────────────────────────┘                    │
├────────────────────────────────────────────────────────────────────────┤
│ [deno] ⬡ emits: horizon                            ◫ control   v2  ▶ run │
│  spur.frontend.slider({ port:"horizon", min:1, max:24, value:12 })       │
│  └─ ◫ FRONTEND CONTROL  [ horizon  ●──────────  12 ]                     │
├────────────────────────────────────────────────────────────────────────┤
│  DAG:  sales → forecast ← horizon        · status: all fresh            │
└────────────────────────────────────────────────────────────────────────┘
```

### 7b. App (Run) mode — wireframe

Code is gone. Only frontend cells remain, arranged as an application. The status strip shows liveness + last cascade latency.

```text
┌────────────────────────────────────────────────────────────────────────┐
│  ◉ Sales Forecast                  (Jute-App)        ⟳ live   ⋯  ─ □ ×  │
├────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│   Horizon (months)    ●──────────────────────   12                       │
│                                                                          │
│   ┌─ Forecast ──────────────────────────────────────────────────────┐   │
│   │      ▁▂▃▅▆▇█▇▆▅▅▆▇█      revenue projection                      │   │
│   │      …re-renders the instant 'horizon' changes…                 │   │
│   └──────────────────────────────────────────────────────────────────┘   │
│                                                                          │
│   ┌─ KPIs ──────────────┐   ┌─ Top regions ─────────────────────────┐   │
│   │  ARR    $4.2M  ▲ 8% │   │  APAC   ████████  EU  █████  US ███   │   │
│   │  Conf.  0.91        │   │                                       │   │
│   └─────────────────────┘   └───────────────────────────────────────┘   │
│                                                                          │
├────────────────────────────────────────────────────────────────────────┤
│  ● python  ● deno    · all cells fresh · last cascade 240 ms             │
└────────────────────────────────────────────────────────────────────────┘
```

**Layout** is driven by frontend-cell order + optional `spur.frontend.props.layout` hints (row/col/span). MVP: vertical stack in document order; later: grid.

**Interaction states per frontend cell:** `idle` → `recomputing` (subtle shimmer while upstream cascades) → `fresh`; `stale` (dashed border, last value dimmed) when an upstream failed; `error` (red strip) with the failing cell named.

### 7c. App (Run) mode — live HTML/JS mockup

The cell below is a **runnable** mockup of App mode. It runs in the **default python3 kernel** (today the notebook is single-kernel; under the target design this would be a Deno *frontend cell* — the HTML/JS body is identical). It emits a self-contained widget via `IPython.display.HTML`.

Dragging the `horizon` slider recomputes the forecast and re-renders the chart + KPIs **live** — a client-side stand-in for the real reactive loop (`source.push → cascade → port.updated → render`). The initial curve is computed in the kernel (so the chart is populated even if the renderer strips `<script>`); the embedded script handles live updates. The forecast is a deterministic mock function of `horizon`; in the real app it would be the `forecast` port produced by an upstream cell.

In [ ]:
// Jute-App — App mode live mockup. Run in a Deno cell.
// Initial curve is computed here (Deno) so the chart renders even if <script> is stripped;
// the embedded script re-renders the chart + KPIs live as the 'horizon' slider moves.

function forecast(h) {
  const pts = [];
  let v = 100;
  for (let i = 0; i < h; i++) { v *= 1.045; pts.push(v * (1 + 0.06 * Math.sin(i / 2.2))); }
  return pts;
}
function chart(pts) {
  const W = 640, H = 180, pad = 10;
  const max = Math.max(...pts), min = Math.min(...pts), span = Math.max(1e-9, max - min);
  const sx = W / Math.max(1, pts.length - 1), sy = (H - 2 * pad) / span;
  const X = (i) => (i * sx).toFixed(1), Y = (i) => (H - pad - (pts[i] - min) * sy).toFixed(1);
  let line = "M" + X(0) + "," + Y(0);
  for (let i = 1; i < pts.length; i++) line += " L" + X(i) + "," + Y(i);
  const area = line + " L" + X(pts.length - 1) + "," + H + " L" + X(0) + "," + H + " Z";
  return { line, area };
}
function kpis(pts) {
  const last = pts[pts.length - 1], first = pts[0];
  return {
    arr: "$" + (last * 0.0345).toFixed(1) + "M",
    growth: "▲ " + ((last / first - 1) * 100).toFixed(0) + "%",
    conf: (0.97 - pts.length * 0.006).toFixed(2),
  };
}

const h0 = 12, p0 = forecast(h0), c0 = chart(p0), k0 = kpis(p0);

const html = String.raw`
<div id="ja-root" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;max-width:720px;border:1px solid #e3e6ea;border-radius:12px;overflow:hidden;box-shadow:0 8px 28px rgba(20,30,50,.10);background:#fff;color:#1c2433">
  <div style="display:flex;align-items:center;gap:8px;padding:10px 14px;background:linear-gradient(180deg,#fafbfc,#eef1f5);border-bottom:1px solid #e3e6ea;font-size:13px">
    <span style="color:#10b981">&#9673;</span><strong>Sales Forecast</strong>
    <span style="color:#8a93a2">(Jute-App)</span>
    <span style="margin-left:auto;display:inline-flex;align-items:center;gap:6px;color:#10b981;font-size:12px"><span style="width:8px;height:8px;border-radius:50%;background:#10b981;display:inline-block"></span> live</span>
  </div>
  <div style="padding:18px">
    <label style="display:flex;align-items:center;gap:12px;font-size:13px;color:#3b4453;margin-bottom:16px">
      <span style="width:130px">Horizon (months)</span>
      <input id="ja-horizon" type="range" min="1" max="24" value="${h0}" style="flex:1;accent-color:#4f46e5">
      <span id="ja-horizon-val" style="width:28px;text-align:right;font-variant-numeric:tabular-nums;font-weight:600">${h0}</span>
    </label>
    <div style="border:1px solid #eceef1;border-radius:10px;padding:12px 14px 6px">
      <div style="font-size:12px;color:#8a93a2;margin-bottom:6px">Forecast &mdash; revenue projection</div>
      <svg id="ja-chart" viewBox="0 0 640 180" width="100%" height="180" preserveAspectRatio="none">
        <path id="ja-area" fill="rgba(79,70,229,.10)" stroke="none" d="${c0.area}"></path>
        <path id="ja-line" fill="none" stroke="#4f46e5" stroke-width="2.5" stroke-linejoin="round" d="${c0.line}"></path>
      </svg>
    </div>
    <div style="display:flex;gap:12px;margin-top:14px">
      <div style="flex:1;border:1px solid #eceef1;border-radius:10px;padding:12px 14px">
        <div style="font-size:11px;color:#8a93a2;text-transform:uppercase;letter-spacing:.04em">KPIs</div>
        <div style="display:flex;justify-content:space-between;margin-top:6px;font-size:14px"><span>ARR</span><strong id="ja-arr">${k0.arr}</strong></div>
        <div style="display:flex;justify-content:space-between;margin-top:4px;font-size:14px"><span>Growth</span><strong id="ja-growth" style="color:#10b981">${k0.growth}</strong></div>
        <div style="display:flex;justify-content:space-between;margin-top:4px;font-size:14px"><span>Confidence</span><strong id="ja-conf">${k0.conf}</strong></div>
      </div>
      <div style="flex:1;border:1px solid #eceef1;border-radius:10px;padding:12px 14px">
        <div style="font-size:11px;color:#8a93a2;text-transform:uppercase;letter-spacing:.04em">Top regions</div>
        <div style="margin-top:10px;display:flex;flex-direction:column;gap:8px;font-size:12px">
          <div style="display:flex;align-items:center;gap:8px"><span style="width:42px;color:#5b6472">APAC</span><span style="height:9px;border-radius:5px;background:#4f46e5;width:74%"></span></div>
          <div style="display:flex;align-items:center;gap:8px"><span style="width:42px;color:#5b6472">EU</span><span style="height:9px;border-radius:5px;background:#6366f1;width:48%"></span></div>
          <div style="display:flex;align-items:center;gap:8px"><span style="width:42px;color:#5b6472">US</span><span style="height:9px;border-radius:5px;background:#818cf8;width:31%"></span></div>
        </div>
      </div>
    </div>
  </div>
  <div style="display:flex;align-items:center;gap:14px;padding:8px 14px;background:#fafbfc;border-top:1px solid #e3e6ea;font-size:12px;color:#8a93a2">
    <span style="color:#10b981">&#9679; python</span><span style="color:#10b981">&#9679; deno</span>
    <span style="margin-left:auto" id="ja-status">all cells fresh &middot; last cascade ${40 + h0 * 7} ms</span>
  </div>
</div>
<script>
(function(){
  var slider = document.getElementById('ja-horizon');
  if (!slider) return;
  function forecast(h){var pts=[],v=100;for(var i=0;i<h;i++){v*=1.045;pts.push(v*(1+0.06*Math.sin(i/2.2)));}return pts;}
  function chart(pts){var W=640,H=180,pad=10;var max=Math.max.apply(null,pts),min=Math.min.apply(null,pts),span=Math.max(1e-9,max-min);var sx=W/Math.max(1,pts.length-1),sy=(H-2*pad)/span;function X(i){return (i*sx).toFixed(1);}function Y(i){return (H-pad-(pts[i]-min)*sy).toFixed(1);}var line='M'+X(0)+','+Y(0);for(var i=1;i<pts.length;i++){line+=' L'+X(i)+','+Y(i);}var area=line+' L'+X(pts.length-1)+','+H+' L'+X(0)+','+H+' Z';return {line:line,area:area};}
  function render(){var h=parseInt(slider.value,10);var pts=forecast(h);var c=chart(pts);document.getElementById('ja-line').setAttribute('d',c.line);document.getElementById('ja-area').setAttribute('d',c.area);document.getElementById('ja-horizon-val').textContent=h;var last=pts[pts.length-1],first=pts[0];document.getElementById('ja-arr').textContent='$'+(last*0.0345).toFixed(1)+'M';document.getElementById('ja-growth').textContent='▲ '+((last/first-1)*100).toFixed(0)+'%';document.getElementById('ja-conf').textContent=(0.97-h*0.006).toFixed(2);document.getElementById('ja-status').textContent='all cells fresh · last cascade '+(40+h*7)+' ms';}
  slider.addEventListener('input', render);
  render();
})();
</script>
`;

await Deno.jupyter.display({ "text/html": html }, { raw: true });


SyntaxError: invalid character '—' (U+2014) (457144185.py, line 244)

## 8. Container lifecycle & supervision

An app must survive a worker dying — show degraded state, restart it, and **re-announce ports from the manifest on reconnect** (manifest-as-truth makes recovery clean).

```mermaid
stateDiagram-v2
  [*] --> Booting: open Jute-App notebook
  Booting --> Wiring: start engine + ipc:// bus broker
  Wiring --> Provisioning: ensure_kernel(python / deno / rust)
  Provisioning --> Hydrating: replay ports from manifest.json
  Hydrating --> Running: render frontend cells (last-known values)
  Running --> Running: source.push / port.updated cascade
  Running --> Degraded: worker heartbeat lost
  Degraded --> Running: supervisor restarts worker + re-announces ports
  Running --> Closing: close notebook
  Closing --> [*]: flush manifest · drop bus socket
```

**Liveness is greenfield.** The Jupyter `heartbeat` socket is currently unused (`driver_zeromq.rs:111`, `let _ = (stdin, heartbeat)`). The app supervisor wires it, plus a bus-level `pong`, to detect dead workers.

**Hydration matters:** because frontend *view* cells re-render from the manifest, an app reopens showing its last computed state instantly, before any kernel is even ready.

## 9. Data / control plane split + invariants

```mermaid
flowchart LR
  subgraph CP["Control plane — ipc:// bus + comm"]
    E1["port.updated / cell.run.request / source.push"]
  end
  subgraph DP["Data plane — disk"]
    D1[("Arrow IPC files")]
    D2[("manifest.json — source of truth")]
  end
  CP -. references only .-> DP
```

**Invariants (non-negotiable):**

1. **`manifest.json` is the single source of truth.** Every bus/comm event is *notify-only*; subscribers re-read the manifest for the authoritative version (`bump_produced_ports_if_unchanged`, `engine.rs:441`). With multiple processes, split-brain risk is higher — this rule prevents it.
2. **No payloads on the bus.** Bulk data stays as Arrow files; the bus carries references + signals. Preserves the polyglot "any process can read a port" property.
3. **Centralized cascade.** The engine alone computes topological order and failure propagation. Decentralized choreography (cells self-firing on upstream bumps) re-introduces reactive *glitches* — diamond DAGs double-fire the join node. (See `docs/rca/…diamond-dag…`.)
4. **Per-notebook scoping.** Bus socket, port store, and container identity all derive from `notebook_id_for_path` (`ports.rs:11`).
5. **Bus auth ≠ kernel auth.** Do not reuse the Jupyter HMAC signing key for the cell bus; `ipc://` filesystem perms (optionally + a per-session token) gate participants.

## 10. Migration — each step ships something usable

| Step | Deliverable | Risk | Touches |
|---|---|---|---|
| **0** | *(prereq)* land per-cell `code_type` routing (multi-kernel per notebook) | — | engine, commands, bindings |
| **1** | **Frontend-cell MVP on existing rails** — a view cell that renders a port and re-renders on `dag.status.changed`. Proves the "app feel" with no new bus. | low | frontend, comm |
| **2** | **`ipc://` bus broker in the engine**, mirroring existing events (observe-only; GUI still works via store). | low (pure addition) | `engine.rs`, new bus module |
| **3** | **spur helper publishes `port.updated`** to the bus (python + deno bootstraps) → engine stops post-run manifest polling for discovery. | med | `ports.rs`, engine |
| **4** | **Frontend control cells publish `source.push`** over the bus → closed reactive loop → *it is an app*. | med | frontend, engine intake |
| **5** | **App mode** UI: title-bar toggle, widgets-only layout, status strip, supervision/heartbeat. | med | frontend, supervisor |
| **6** | *(optional, later)* embeddable / headless runner — same engine + bus, no GUI. | — | packaging |

**Sequencing rule:** steps 1–2 are independent and low-risk; do them first to de-risk the comm + bus substrate before the reactive loop (4) depends on both.

## 11. Risks & open questions

**Risks**
- **PUB/SUB drops on slow subscribers** (ZeroMQ high-water mark). For `cell.run.request` use ROUTER/DEALER with acks, or have every subscriber reconcile against the manifest on connect. Do not trust fire-and-forget PUB for run requests.
- **Glitches** if cascade ever decentralizes — hold invariant #3.
- **Comm protocol completeness** — `CommOpen`/`CommMessage` exist but the full open/msg/close + frontend widget registry may be only partially wired; step 1 must verify the round-trip.
- **Layout scope creep** — App mode layout can balloon into a full UI builder. MVP = document-order vertical stack; defer grid/freeform.

**Open questions**
1. **Widget library**: build a small native set (chart/table/slider/form) vs. adopt an existing protocol (anywidget / ipywidgets)? Adopting buys ecosystem but couples to ipywidgets comm semantics.
2. **Multi-client**: can two windows open the same Jute-App? If yes, the bus broker must fan out comm state to N frontends (the manifest already supports it; comm session routing does not yet).
3. **Persistence of control state**: is a slider's last value part of the document, a port, or ephemeral session state? Proposal: it is a port (`horizon`), so it persists in the manifest and the app reopens where you left it.
4. **Security**: for any future shared/remote deployment, `ipc://` + filesystem perms is local-only — a remote story needs a gateway, out of scope here.

---

*Next: on approval, convert §10 into an implementation plan (writing-plans), starting with steps 1–2.*

In [ ]:
# SPUR datasource setup cell v1
# This cell is managed by SPUR. Re-run it after datasource changes.
import duckdb

_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
duckdb.sql(f"LOAD '{_SPUR_DUCKDB_EXTENSION_SQL}'")

